In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
!pip install -q transformers datasets accelerate scikit-learn

In [ ]:
import pandas as pd

In [ ]:
fake_df = pd.read_csv("Fake.csv", engine='python', on_bad_lines='skip')
true_df = pd.read_csv("True.csv", engine='python', on_bad_lines='skip')

FileNotFoundError: [Errno 2] No such file or directory: 'Fake.csv'

In [ ]:
print(fake_df.head())

In [ ]:
print(true_df.head())

In [ ]:
print(fake_df.shape)
print(true_df.shape)

In [ ]:
print(fake_df.columns)

In [ ]:
print("Fake news subjects:")
print(fake_df['subject'].value_counts())

print("\nReal news subjects:")
print(true_df['subject'].value_counts())

In [ ]:
print("Fake news subjects:")
print(fake_df['subject'].value_counts())

print("\nReal news subjects:")
print(true_df['subject'].value_counts())

In [ ]:
print("Fake unique subjects:")
print(fake_df['subject'].value_counts(dropna=False))

print("\nReal unique subjects:")
print(true_df['subject'].value_counts(dropna=False))

In [ ]:
print("Fake unique subjects:")
print(fake_df['subject'].value_counts(dropna=False))

print("\nReal unique subjects:")
print(true_df['subject'].value_counts(dropna=False))

In [ ]:
fake_df['subject'].value_counts(dropna=False)

In [ ]:
true_df['subject'].value_counts(dropna=False)

In [ ]:
true_df['subject'].value_counts(dropna=False)

In [ ]:
fake_df['label'] = 1
true_df['label'] = 0

In [ ]:
print(fake_df['label'].value_counts())
print(true_df['label'].value_counts())

In [ ]:
df = pd.concat([fake_df, true_df], ignore_index=True)

print(df.shape)

In [ ]:
print(df.shape)

print("\nLabel distribution:")
print(df['label'].value_counts())

In [ ]:
df = pd.concat([fake_df, true_df], ignore_index=True)

print(df.shape)

print("\nLabel distribution:")
print(df['label'].value_counts())

In [ ]:
duplicates = df['text'].duplicated().sum()

print("Duplicate articles:", duplicates)

In [ ]:
duplicates = df['text'].duplicated().sum()

print("Duplicate articles:", duplicates)

In [ ]:
df = df.drop_duplicates(subset='text', keep='first').reset_index(drop=True)

print("New dataset shape:", df.shape)
print("Remaining duplicates:", df['text'].duplicated().sum())

In [ ]:
print(df['label'].value_counts())
print()
print(df['label'].value_counts(normalize=True))

In [ ]:
df['content'] = df['title'].fillna('') + ' ' + df['text'].fillna('')

In [ ]:
df = df[['content', 'label']]

In [ ]:
print(df.head())
print(df.shape)

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df['content'].tolist(),
    df['label'].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=df['label']
)

In [ ]:
print("Training samples:", len(train_texts))
print("Validation samples:", len(val_texts))

print("\nTraining distribution:")
print(pd.Series(train_labels).value_counts(normalize=True))

print("\nValidation distribution:")
print(pd.Series(val_labels).value_counts(normalize=True))

In [ ]:
from transformers import DistilBertTokenizerFast

tokenizer = DistilBertTokenizerFast.from_pretrained(
    "distilbert-base-uncased"
)

In [ ]:
sample = train_texts[0]

print(sample[:500])

In [ ]:
encoded = tokenizer(
    sample,
    truncation=True,
    padding="max_length",
    max_length=256,
    return_tensors="pt"
)

In [ ]:
print(encoded.keys())
print(encoded["input_ids"].shape)
print(encoded["input_ids"][0][:20])
print(encoded["attention_mask"][0][:20])

In [ ]:
import torch
from torch.utils.data import Dataset

In [ ]:
class FakeNewsDataset(Dataset):

    def __init__(self, texts, labels, tokenizer, max_length=256):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):

        text = self.texts[idx]
        label = self.labels[idx]

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(label, dtype=torch.long)
        }

In [ ]:
train_dataset = FakeNewsDataset(
    train_texts,
    train_labels,
    tokenizer,
    max_length=256
)

val_dataset = FakeNewsDataset(
    val_texts,
    val_labels,
    tokenizer,
    max_length=256
)

In [ ]:
sample_item = train_dataset[0]

print(sample_item.keys())
print("input_ids shape:", sample_item["input_ids"].shape)
print("attention_mask shape:", sample_item["attention_mask"].shape)
print("label:", sample_item["labels"])

In [ ]:
from transformers import DistilBertForSequenceClassification

model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

In [ ]:
print(model)

In [ ]:
print("Number of parameters:", model.num_parameters())

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.to(device)

print("Using device:", device)

In [ ]:
from torch.utils.data import DataLoader

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False
)

In [ ]:
batch = next(iter(train_loader))

print(batch.keys())
print("Input IDs shape:", batch["input_ids"].shape)
print("Attention mask shape:", batch["attention_mask"].shape)
print("Labels shape:", batch["labels"].shape)

In [ ]:
batch = {
    key: value.to(device)
    for key, value in batch.items()
}

In [ ]:
model.eval()

with torch.no_grad():
    outputs = model(
        input_ids=batch["input_ids"],
        attention_mask=batch["attention_mask"]
    )

In [ ]:
print(outputs)

In [ ]:
print("Logits shape:", outputs.logits.shape)

In [ ]:
from torch.optim import AdamW

optimizer = AdamW(
    model.parameters(),
    lr=2e-5
)

print("Optimizer created successfully")

In [ ]:
model.train()

# Get one batch
batch = next(iter(train_loader))

# Move batch to GPU
batch = {
    key: value.to(device)
    for key, value in batch.items()
}

# Clear old gradients
optimizer.zero_grad()

# Forward pass
outputs = model(
    input_ids=batch["input_ids"],
    attention_mask=batch["attention_mask"],
    labels=batch["labels"]
)

# Get loss
loss = outputs.loss

# Calculate gradients
loss.backward()

# Update model weights
optimizer.step()

print("Training step completed")
print("Loss:", loss.item())

In [ ]:
def evaluate_model(model, val_loader, device):

    model.eval()

    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():

        for batch in val_loader:

            batch = {
                key: value.to(device)
                for key, value in batch.items()
            }

            outputs = model(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"],
                labels=batch["labels"]
            )

            loss = outputs.loss
            predictions = torch.argmax(outputs.logits, dim=1)

            total_loss += loss.item()

            correct += (predictions == batch["labels"]).sum().item()
            total += batch["labels"].size(0)

    average_loss = total_loss / len(val_loader)
    accuracy = correct / total

    return average_loss, accuracy

In [ ]:
val_loss, val_accuracy = evaluate_model(
    model,
    val_loader,
    device
)

print("Validation Loss:", val_loss)
print("Validation Accuracy:", val_accuracy)

In [ ]:
from transformers import DistilBertForSequenceClassification

model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

model = model.to(device)

# Recreate optimizer for the fresh model
optimizer = AdamW(
    model.parameters(),
    lr=2e-5
)

print("Fresh model and optimizer ready")

In [ ]:
import time

num_epochs = 2

for epoch in range(num_epochs):

    model.train()

    total_loss = 0
    start_time = time.time()

    for step, batch in enumerate(train_loader):

        # Move batch to GPU
        batch = {
            key: value.to(device)
            for key, value in batch.items()
        }

        # Clear previous gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            labels=batch["labels"]
        )

        # Get loss
        loss = outputs.loss

        # Backpropagation
        loss.backward()

        # Update model weights
        optimizer.step()

        total_loss += loss.item()

        # Show progress every 100 batches
        if (step + 1) % 100 == 0:
            print(
                f"Epoch {epoch + 1}/{num_epochs} | "
                f"Step {step + 1}/{len(train_loader)} | "
                f"Loss: {loss.item():.4f}"
            )

    average_train_loss = total_loss / len(train_loader)

    # Validate after each epoch
    val_loss, val_accuracy = evaluate_model(
        model,
        val_loader,
        device
    )

    epoch_time = time.time() - start_time

    print("\n" + "=" * 60)
    print(f"Epoch {epoch + 1} completed")
    print(f"Training Loss: {average_train_loss:.4f}")
    print(f"Validation Loss: {val_loss:.4f}")
    print(f"Validation Accuracy: {val_accuracy:.4f}")
    print(f"Time: {epoch_time / 60:.2f} minutes")
    print("=" * 60 + "\n")

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

model.eval()

all_predictions = []
all_labels = []

with torch.no_grad():

    for batch in val_loader:

        batch = {
            key: value.to(device)
            for key, value in batch.items()
        }

        outputs = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"]
        )

        predictions = torch.argmax(outputs.logits, dim=1)

        all_predictions.extend(predictions.cpu().numpy())
        all_labels.extend(batch["labels"].cpu().numpy())


accuracy = accuracy_score(all_labels, all_predictions)
precision = precision_score(all_labels, all_predictions)
recall = recall_score(all_labels, all_predictions)
f1 = f1_score(all_labels, all_predictions)

print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)

print("\nClassification Report:")
print(
    classification_report(
        all_labels,
        all_predictions,
        target_names=["REAL", "FAKE"]
    )
)

print("Confusion Matrix:")
print(confusion_matrix(all_labels, all_predictions))

In [ ]:
print("Total dataframe rows:", len(df))

print("Train dataset:", len(train_dataset))
print("Validation dataset:", len(val_dataset))

print("Train loader batches:", len(train_loader))
print("Validation loader batches:", len(val_loader))

print("Batch size:", train_loader.batch_size)

In [ ]:
print("Fake.csv rows:", len(fake_df))
print("True.csv rows:", len(true_df))

print("\nFake label count:")
print(fake_df["label"].value_counts())

print("\nTrue label count:")
print(true_df["label"].value_counts())

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

cm = confusion_matrix(all_labels, all_predictions)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["REAL", "FAKE"]
)

disp.plot()

plt.title("DistilBERT Fake News Detection - Confusion Matrix")
plt.show()

In [ ]:
def predict_news(text):

    model.eval()

    encoding = tokenizer(
        text,
        truncation=True,
        padding="max_length",
        max_length=256,
        return_tensors="pt"
    )

    input_ids = encoding["input_ids"].to(device)
    attention_mask = encoding["attention_mask"].to(device)

    with torch.no_grad():

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

    probabilities = torch.softmax(outputs.logits, dim=1)

    prediction = torch.argmax(probabilities, dim=1).item()

    confidence = probabilities[0][prediction].item()

    if prediction == 0:
        label = "REAL"
    else:
        label = "FAKE"

    return label, confidence

In [ ]:
test_news = """
The Indian government announced a new policy that will provide
additional support for technology startups across the country.
"""

label, confidence = predict_news(test_news)

print("Prediction:", label)
print("Confidence:", f"{confidence * 100:.2f}%")

In [ ]:
import random

model.eval()

for i in random.sample(range(len(val_texts)), 5):

    text = val_texts[i]
    actual_label = val_labels[i]

    predicted_label, confidence = predict_news(text)

    actual_name = "REAL" if actual_label == 0 else "FAKE"

    print("=" * 70)
    print("Actual     :", actual_name)
    print("Prediction :", predicted_label)
    print("Confidence :", f"{confidence * 100:.2f}%")
    print("Text       :", text[:300], "...")

In [ ]:
model.save_pretrained("fake_news_distilbert")
tokenizer.save_pretrained("fake_news_distilbert")

print("Model and tokenizer saved successfully!")

In [ ]:
import os

print(os.listdir("/content/fake_news_distilbert"))

In [ ]:
import shutil
import os

source = "/content/fake_news_distilbert"
destination = "/content/drive/MyDrive/fake_news_distilbert"

shutil.copytree(source, destination, dirs_exist_ok=True)

print("Model permanently saved to:")
print(destination)

In [ ]:
def predict_news(text):

    model.eval()

    encoding = tokenizer(
        text,
        truncation=True,
        padding="max_length",
        max_length=256,
        return_tensors="pt"
    )

    input_ids = encoding["input_ids"].to(device)
    attention_mask = encoding["attention_mask"].to(device)

    with torch.no_grad():

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

    probabilities = torch.softmax(outputs.logits, dim=1)[0]

    real_probability = probabilities[0].item()
    fake_probability = probabilities[1].item()

    prediction = "FAKE" if fake_probability > real_probability else "REAL"

    return {
        "prediction": prediction,
        "real_probability": real_probability,
        "fake_probability": fake_probability
    }

In [ ]:
news = """
Scientists have announced a new technology that could significantly
improve the efficiency of solar panels.
"""

result = predict_news(news)

print("Prediction:", result["prediction"])
print("REAL probability:", f"{result['real_probability'] * 100:.2f}%")
print("FAKE probability:", f"{result['fake_probability'] * 100:.2f}%")

In [ ]:
from IPython.display import display
import ipywidgets as widgets

news_input = widgets.Textarea(
    value='',
    placeholder='Paste the news article here...',
    description='News:',
    layout=widgets.Layout(width='100%', height='250px')
)

predict_button = widgets.Button(
    description='Predict',
    button_style='primary'
)

output = widgets.Output()

def on_predict_clicked(b):

    with output:
        output.clear_output()

        text = news_input.value.strip()

        if not text:
            print("Please enter some news text.")
            return

        result = predict_news(text)

        print("Prediction:", result["prediction"])
        print(
            "REAL probability:",
            f"{result['real_probability'] * 100:.2f}%"
        )
        print(
            "FAKE probability:",
            f"{result['fake_probability'] * 100:.2f}%"
        )

predict_button.on_click(on_predict_clicked)

display(news_input)
display(predict_button)
display(output)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1, 2),
    stop_words="english"
)

X_train_tfidf = tfidf.fit_transform(train_texts)
X_val_tfidf = tfidf.transform(val_texts)

print("Training TF-IDF shape:", X_train_tfidf.shape)
print("Validation TF-IDF shape:", X_val_tfidf.shape)

In [ ]:
from sklearn.linear_model import LogisticRegression

lr_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

print("Training Logistic Regression...")

lr_model.fit(
    X_train_tfidf,
    train_labels
)

print("Logistic Regression training completed!")

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# Make predictions
lr_predictions = lr_model.predict(X_val_tfidf)

# Calculate metrics
lr_accuracy = accuracy_score(val_labels, lr_predictions)
lr_precision = precision_score(val_labels, lr_predictions)
lr_recall = recall_score(val_labels, lr_predictions)
lr_f1 = f1_score(val_labels, lr_predictions)

print("Logistic Regression Results")
print("=" * 40)

print("Accuracy :", lr_accuracy)
print("Precision:", lr_precision)
print("Recall   :", lr_recall)
print("F1 Score :", lr_f1)

print("\nClassification Report:")
print(
    classification_report(
        val_labels,
        lr_predictions,
        target_names=["REAL", "FAKE"]
    )
)

print("Confusion Matrix:")
print(confusion_matrix(val_labels, lr_predictions))


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Use a smaller TF-IDF representation for this audit
audit_vectorizer = TfidfVectorizer(
    max_features=20000,
    stop_words="english"
)

X_train_audit = audit_vectorizer.fit_transform(train_texts)
X_val_audit = audit_vectorizer.transform(val_texts)

print("Train matrix:", X_train_audit.shape)
print("Validation matrix:", X_val_audit.shape)

# Find the most similar training article for each validation article
similarity = cosine_similarity(
    X_val_audit,
    X_train_audit
)

max_similarity = similarity.max(axis=1)

print("\nSimilarity statistics:")
print("Minimum:", max_similarity.min())
print("Mean   :", max_similarity.mean())
print("Maximum:", max_similarity.max())

print("\nVery high similarity:")
print(">= 0.90:", (max_similarity >= 0.90).sum())
print(">= 0.95:", (max_similarity >= 0.95).sum())
print(">= 0.99:", (max_similarity >= 0.99).sum())

In [ ]:
import numpy as np

# Find the training article with maximum similarity
best_train_indices = similarity.argmax(axis=1)

# Create a small audit table
audit_results = []

for val_idx, train_idx in enumerate(best_train_indices):

    sim = max_similarity[val_idx]

    if sim >= 0.99:

        audit_results.append({
            "validation_index": val_idx,
            "training_index": train_idx,
            "similarity": sim,
            "validation_label": val_labels[val_idx],
            "training_label": train_labels[train_idx]
        })

audit_df = pd.DataFrame(audit_results)

print("Articles with similarity >= 0.99:")
print(len(audit_df))

print("\nLabel combinations:")
print(
    audit_df.groupby(
        ["training_label", "validation_label"]
    ).size()
)

print("\nFirst 20:")
print(audit_df.head(20))

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold
import numpy as np
import pandas as pd

# Combine the original train and validation text/labels
all_texts = train_texts + val_texts
all_labels = train_labels + val_labels

n_train = len(train_texts)
n_total = len(all_texts)

# Start with every article in its own group
parent = list(range(n_total))


# ---------- Union-Find functions ----------

def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x


def union(a, b):
    root_a = find(a)
    root_b = find(b)

    if root_a != root_b:
        parent[root_b] = root_a


# ---------- Connect highly similar articles ----------

for val_idx, train_idx in zip(
    np.where(max_similarity >= 0.99)[0],
    best_train_indices[max_similarity >= 0.99]
):

    # Convert validation index to combined-data index
    global_val_idx = n_train + val_idx

    # Training index is already a combined-data index
    global_train_idx = train_idx

    union(global_val_idx, global_train_idx)


# Create group IDs
group_map = {}

groups = []

for i in range(n_total):

    root = find(i)

    if root not in group_map:
        group_map[root] = len(group_map)

    groups.append(group_map[root])


groups = np.array(groups)

print("Total articles:", n_total)
print("Number of similarity groups:", len(np.unique(groups)))

print("Articles connected to another article:",
      n_total - sum(
          np.bincount(groups) == 1
      ))

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

sgkf = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Get the first 80/20 split
train_idx, val_idx = next(
    sgkf.split(
        all_texts,
        all_labels,
        groups=groups
    )
)

robust_train_texts = [all_texts[i] for i in train_idx]
robust_val_texts = [all_texts[i] for i in val_idx]

robust_train_labels = [all_labels[i] for i in train_idx]
robust_val_labels = [all_labels[i] for i in val_idx]

print("Robust training articles:", len(robust_train_texts))
print("Robust validation articles:", len(robust_val_texts))

print("\nTraining labels:")
print(pd.Series(robust_train_labels).value_counts())

print("\nValidation labels:")
print(pd.Series(robust_val_labels).value_counts())


In [ ]:
robust_train_dataset = FakeNewsDataset(
    robust_train_texts,
    robust_train_labels,
    tokenizer,
    max_length=256
)

robust_val_dataset = FakeNewsDataset(
    robust_val_texts,
    robust_val_labels,
    tokenizer,
    max_length=256
)

robust_train_loader = DataLoader(
    robust_train_dataset,
    batch_size=16,
    shuffle=True
)

robust_val_loader = DataLoader(
    robust_val_dataset,
    batch_size=16,
    shuffle=False
)

print("Robust datasets created!")
print("Training batches:", len(robust_train_loader))
print("Validation batches:", len(robust_val_loader))

In [ ]:
from transformers import DistilBertForSequenceClassification
from torch.optim import AdamW

# Fresh model
robust_model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

robust_model = robust_model.to(device)

# Fresh optimizer
robust_optimizer = AdamW(
    robust_model.parameters(),
    lr=2e-5
)

print("Fresh robust model ready")

In [ ]:
import time

num_epochs = 2

for epoch in range(num_epochs):

    robust_model.train()

    total_loss = 0
    start_time = time.time()

    for step, batch in enumerate(robust_train_loader):

        batch = {
            key: value.to(device)
            for key, value in batch.items()
        }

        robust_optimizer.zero_grad()

        outputs = robust_model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            labels=batch["labels"]
        )

        loss = outputs.loss

        loss.backward()

        robust_optimizer.step()

        total_loss += loss.item()

        if (step + 1) % 100 == 0:
            print(
                f"Epoch {epoch + 1}/{num_epochs} | "
                f"Step {step + 1}/{len(robust_train_loader)} | "
                f"Loss: {loss.item():.4f}"
            )

    average_train_loss = total_loss / len(robust_train_loader)

    # Validation
    robust_model.eval()

    correct = 0
    total = 0
    val_total_loss = 0

    with torch.no_grad():

        for batch in robust_val_loader:

            batch = {
                key: value.to(device)
                for key, value in batch.items()
            }

            outputs = robust_model(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"],
                labels=batch["labels"]
            )

            val_total_loss += outputs.loss.item()

            predictions = torch.argmax(
                outputs.logits,
                dim=1
            )

            correct += (
                predictions == batch["labels"]
            ).sum().item()

            total += batch["labels"].size(0)

    average_val_loss = (
        val_total_loss / len(robust_val_loader)
    )

    val_accuracy = correct / total

    epoch_time = time.time() - start_time

    print("\n" + "=" * 60)
    print(f"Epoch {epoch + 1} completed")
    print(f"Training Loss: {average_train_loss:.4f}")
    print(f"Validation Loss: {average_val_loss:.4f}")
    print(f"Validation Accuracy: {val_accuracy:.4f}")
    print(f"Time: {epoch_time / 60:.2f} minutes")
    print("=" * 60 + "\n")